## 🔥 <span style='text-decoration: double underline; color:rgb(10,110,217)'>**PyTorch: Fundamentals**</span>

**Author:** Ángel Fernández Caravaca

**GitHub:** [@afdezcaravaca](https://github.com/afdezcaravaca)

**Date:** March 2026

### 📑 <span style='color:rgb(10,110,217)'><u>**Introduction to PyTorch**</u></span>

#### ❓<span style='color:rgb(10,110,217)'><u>**What is PyTorch?**</u></span>
PyTorch is an open-source deep learning framework originally developed by Meta's AI Research lab (FAIR). It has become the dominant tool in both academic research and production environments, thanks to its **Python-first design philosophy**, its flexibility for rapid prototyping, and its performance for large-scale training.

At its core, PyTorch is built around two foundational abstractions:

- **`torch.Tensor`** — the primary data structure, functionally equivalent to NumPy's `ndarray` but engineered for hardware-accelerated parallel computation (NVIDIA GPUs via CUDA, Apple Silicon via MPS).
- **`torch.autograd`** — an automatic differentiation engine that tracks all tensor operations and computes gradients through backpropagation, which is the mathematical backbone of training neural networks.

#### 🌊 <span style='color:rgb(10,110,217)'><u>**The Architectural Advantage: Dynamic Computational Graphs**</u></span>

The most defining architectural feature of PyTorch is its implementation of **Dynamic Computational Graphs** (also known as the *Define-by-Run* paradigm).

Unlike older frameworks (e.g., early TensorFlow 1.x) that required constructing a static, pre-compiled computation graph before any data flowed through it, PyTorch builds the graph **on the fly**, operation by operation, as Python code executes.

This design offers critical engineering advantages:
<div align='center'>

| Feature | Static Graph | Dynamic Graph (PyTorch) |
|---|---|---|
| Debugging | Requires special tools | Native Python debuggers (`pdb`, `print`) |
| Control flow | Must be encoded in the graph DSL | Standard Python `if`, `for`, `while` |
| Interoperability | Often isolated | Seamless with NumPy, Pandas, SciPy |
| Flexibility | Limited | Ideal for variable-length sequences, recursive models |
</div>

* **Native Debugging:** Because execution is immediate, developers can use standard Python debuggers (like `pdb`) and standard `print` statements to inspect tensor values and gradients at any arbitrary step.
* **Control Flow Flexibility:** It natively supports standard Python control flow (`if`, `for`, `while`) within the model's forward pass, making it structurally ideal for complex architectures like recurrent networks with variable-length sequences or recursive structures.
* **Ecosystem Interoperability:** It avoids creating an isolated sub-language, ensuring seamless zero-copy interoperability with libraries like NumPy, Pandas, and SciPy.


#### 🧱 <span style='color:rgb(10,110,217)'><u>**Tensors and Ranks:**</u></span>
A **tensor** is a generalization of scalars, vectors, and matrices to an arbitrary number of dimensions. In PyTorch, every piece of data is represented as a `torch.Tensor`. The **rank** of a tensor refers to its number of dimensions (`ndim`):

- **Rank 0 Tensor** $\rightarrow$ () $\rightarrow$ Dimensionless, a mathematical scalar.
- **Rank 1 Tensor** $\rightarrow$ (N,) $\rightarrow$ A 1D flat array/vector, without implicit row/column orientation.
- **Rank 2 Tensor** $\rightarrow$ (H, W) or (Batch, Features) $\rightarrow$ A 2D matrix, including explicit (1, N) row and (N, 1) column vectors.
- **Rank 3 Tensor** $\rightarrow$ (C, H, W) $\rightarrow$ A 3D tensor, such as a multi-channel spatial matrix like a single image, or a temporal sequence of embeddings.
- **Rank 4 Tensor** $\rightarrow$ (B, C, H, W) $\rightarrow$ A 4D tensor, which usually represents a batched collection of 3D tensors, which is the standard input requirement for most neural network layers.

<div align='center'>
<img src="https://cdn-images-1.medium.com/max/2000/1*_D5ZvufDS38WkhK9rK32hQ.jpeg" alt="Tensors" width="800">

</div>

Understanding tensor rank is critical because most neural network layers impose strict dimensional requirements. For instance, `Conv2d` always expects input of rank 4.

#### 🎫 <span style='color:rgb(10,110,217)'><u>**Data types**</u></span>

PyTorch supports a rich type system for tensors. The **numeric suffix** (8, 16, 32, 64) denotes the number of bits used to store each value:

- **Larger bit-width** → greater precision or range, but higher memory consumption and slower computation.
- **Smaller bit-width** → less memory, faster throughput (especially on GPUs), at the cost of numerical precision.

The **`u` prefix** (e.g., `uint8`) denotes **unsigned** types — restricted to zero and positive integers, which doubles the representable positive range relative to their signed counterpart.

**PyTorch defaults:** `float32` for floating-point tensors, `int64` for integer tensors. NumPy, by contrast, defaults to `float64`, which can cause silent precision mismatches when interoperating between the two.

Common types:

<div align='center'>

| Type | Bits | Use case |
|---|---|---|
| `torch.bool` | 1 | Boolean masks |
| `torch.int8 / uint8` | 8 | Quantized models, image pixel values |
| `torch.float16 / bfloat16` | 16 | Mixed-precision training |
| `torch.float32` | 32 | Default for NN weights |
| `torch.float64` | 64 | High-precision scientific computation |
</div>

Type casting is performed via `.to(dtype=...)` or convenience methods like `.float()`, `.int()`, `.bool()`.


#### 🔪 <span style='color:rgb(10,110,217)'><u>**Tensor Indexing and Slicing**</u></span>

PyTorch follows NumPy-style indexing conventions. Tensors can be accessed using standard Python slice notation `[start:stop:step]` along any dimension.

Key concepts:

- **Row selection:** `tensor[i, :]` selects the entire i-th row.
- **Column selection:** `tensor[:, j]` selects the entire j-th column.
- **Sub-tensor extraction:** `tensor[-2:, -2:]` extracts the bottom-right 2×2 block using negative indexing.
- **Boolean masking:** A boolean tensor of the same shape can be used to filter elements satisfying a condition (`tensor[tensor > 5]`).
- **`torch.where(condition)`:** Returns the indices of elements satisfying a condition, analogous to `np.where`.
- **`torch.masked_select(tensor, mask)`:** Extracts a flat 1D tensor of all elements where the mask is `True`.


#### 📐<span style='color:rgb(10,110,217)'><u>**Shape Manipulation**</u></span>

Reshaping is a fundamental operation in deep learning pipelines. PyTorch provides several tools:

##### `reshape` / `view`
Both change the logical shape of a tensor without modifying the underlying data. The total number of elements must be preserved.

- **`.view()`** requires the tensor to be **contiguous** in memory (i.e., its logical layout matches its physical memory layout).
- **`.reshape()`** is more permissive — it calls `.contiguous()` internally if needed, making it safer for general use.

##### `squeeze` / `unsqueeze`
These operations add or remove dimensions of size 1 without changing the data volume.

- **`.unsqueeze(dim)`** — injects a new axis at position `dim`.
- **`.squeeze(dim)`** — removes the axis at position `dim` if its size is 1.

Their primary use is ensuring **dimensional compatibility** with neural network layers and broadcasting rules. For example, a single image of shape `(C, H, W)` must be unsqueezed to `(1, C, H, W)` before being passed to a `Conv2d` layer.

##### `permute` / `transpose`
- **`.permute(*dims)`** — reorders all dimensions simultaneously using their positional indices (generalization of transpose).
- **`.T`** — reverses all dimensions (only reliable and non-deprecated for 2D tensors).

> ⚠️ Both `.permute()` and `.T` on tensors with rank > 2 produce **non-contiguous** tensors (they reorder strides without moving bytes), which will cause `.view()` to raise a `RuntimeError`. The fix is `.contiguous().view(...)`.


#### 💾 <span style='color:rgb(10,110,217)'><u>**Memory: Views, Clones, and Contiguity**</u></span>

Understanding memory ownership is critical for correctness and debugging.
<div align='center'>

| Operation | Copies memory? | Shares underlying data? |
|---|---|---|
| `.view()` | No | Yes — mutations propagate |
| `.reshape()` | Sometimes | Usually yes |
| `.clone()` | Yes | No — fully independent copy |
| `.numpy()` | No | Yes — zero-copy bridge |
| `torch.from_numpy()` | No | Yes — zero-copy bridge |
| `torch.tensor(np_array)` | Yes | No — strict copy |
</div>
The **zero-copy bridge** between PyTorch and NumPy (via `.numpy()` and `torch.from_numpy()`) means that modifying the NumPy array also mutates the tensor, and vice versa. Use `.clone()` before converting when independence is required.

**Contiguity:** A tensor is contiguous when its elements are stored sequentially in memory in the same order as their logical indexing. Operations like `.T` and `.permute()` alter strides (the step size between elements in memory) without physically moving data, producing non-contiguous tensors.


#### 🔣 <span style='color:rgb(10,110,217)'><u>**Aggregation and Reduction Operations**</u></span>

PyTorch provides a rich set of aggregation functions: `.sum()`, `.mean()`, `.std()`, `.min()`, `.max()`, `.argmin()`, `.argmax()`.

A key parameter is `dim`, which specifies the axis *along which* the reduction is applied:

- For a tensor of shape `(B, C, H, W)`:
  - `tensor.sum(dim=0)` collapses the batch dimension → result shape `(C, H, W)`.
  - `tensor.mean(dim=1)` collapses the channel dimension → result shape `(B, H, W)`.

The `keepdim=True` argument preserves the reduced dimension as a size-1 axis, which is essential for **broadcasting** in subsequent operations (e.g., standardizing by subtracting the per-row mean without introducing shape mismatches).


#### 📶 <span style='color:rgb(10,110,217)'><u>**Broadcasting**</u></span>

Broadcasting is a mechanism that allows PyTorch to perform arithmetic operations on tensors of **different but compatible shapes** without explicit data replication.

Two shapes are broadcast-compatible if, aligned from the right, each pair of dimensions is either equal or one of them is 1. For example:

```
(4, 3)  +  (1, 3)  →  (4, 3)   ✓ — the (1, 3) tensor is "virtually" tiled across rows
(4, 1)  +  (1, 3)  →  (4, 3)   ✓ — outer product-like expansion
(4, 3)  +  (4,)    →  ERROR     ✗ — shapes do not align from the right
```

Broadcasting avoids explicit loops and memory duplication, enabling **vectorized code** that is both concise and computationally efficient. A practical example is generating a multiplication table as a `(10, 10)` tensor via the outer product of a `(10, 1)` column vector and a `(1, 10)` row vector.


#### 👷<span style='color:rgb(10,110,217)'><u>**Concatenation vs. Stacking**</u></span>

Both operations combine multiple tensors, but they differ fundamentally:

<div align='center'>

| Operation | Effect on shape | New dimension? |
|---|---|---|
| `torch.cat([t1, t2], dim=d)` | Grows an existing dimension | No |
| `torch.stack([t1, t2], dim=d)` | Inserts a new dimension at position `d` | Yes |
</div>

For tensors of shape `(2, 4)`:
- `torch.cat([...], dim=0)` → `(6, 4)` — stacks rows.
- `torch.cat([...], dim=1)` → `(2, 12)` — stacks columns.
- `torch.stack([...], dim=0)` → `(3, 2, 4)` — creates a new leading dimension.

`torch.cat` requires matching shapes on all dimensions except the concatenation axis. Tensors of different ranks cannot be concatenated directly — they must first be aligned using `unsqueeze`.


#### 🥷 <span style='color:rgb(10,110,217)'><u>**Sparse Tensors**</u></span>

In many real-world scenarios (natural language processing, graph neural networks, recommendation systems), data matrices are overwhelmingly populated with zeros. Storing these as dense tensors wastes memory and computation.

PyTorch supports **sparse COO tensors** (Coordinate format), which store only the indices and values of non-zero elements:

```
Dense:  [[0, 0, 0], [0, 5, 0], [0, 0, 3]]
Sparse: indices=[[1,2],[1,2]], values=[5, 3]
```

**When are sparse tensors beneficial?**

- **Memory efficiency:** Relevant only at high sparsity ratios (typically > 90% zeros), since the index storage overhead must be offset by the savings on zero values.
- **Computational speedup:** Specialized sparse operators (e.g., `torch.smm`) skip arithmetic on zero entries entirely.

#### 🌱<span style='color:rgb(10,110,217)'><u>**Reproducibility and Random Seeds**</u></span>

Training neural networks involves stochastic operations (random weight initialization, data shuffling, dropout). Without controlling randomness, results cannot be reproduced across runs, making debugging, comparison, and publication unreliable.

`torch.manual_seed(seed)` initializes the pseudo-random number generator (PRNG) to a deterministic state. Any call to a random function after setting the seed will always produce the same sequence of values.

**Why is reproducibility critical?**

1. **Debugging:** Isolate whether a training run's behavior is due to code changes or random variation.
2. **Fair comparison:** Ensure two model variants start from identical conditions.
3. **Scientific integrity:** Allow others to replicate published results exactly.

> For full reproducibility in multi-GPU environments, additional steps are required: seeding `numpy`, `random`, and setting `torch.backends.cudnn.deterministic = True`.

---

### ⚒️ <span style='color:rgb(10,110,217)'><u>**Installation**</u></span>
Create your conda/python enviroment for PyTorch and follow these [instructions](https://pytorch.org/get-started/locally/). Notice that you must:

- **Python version $\geq$ 3.10**
- **GPU or CPU:** if you want to use PyTorch on your GPU you must install it with CUDA support. To identify the maximum CUDA version your current drivers support, run nvidia-smi in the terminal and check the "CUDA Version" field in the upper right corner.

---

### 🐤 <span style='color:rgb(10,110,217)'><u>**Level 0: Baby steps**</u></span>
Use the [official documentation](https://docs.pytorch.org/docs/stable/index.html) while solving the exercises. These exercises can be solving only wiht `import torch`.

#####  <span style='color:rgb(10,110,217)'>🔢 **Exercise 0:**</span>
> **Check your PyTorch version and available CUDA devices (names).**

In [ ]:
import torch 

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 1:**</span>
> **Create a zeros tensor of shape `(3,4)`, a ones tensor of shape `(2,2,2)`, and a tensor with random values of shape `(5,5)`.**

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 2:**</span>

> **Given to tensors `a=[1,2,3]` and `b = [4,5,6]`, compute their sum, difference, element-wise product and matrix product.**

**Note:** torch.matmul is more versatile than torch.dot. While torch.dot strictly computes the dot product of two 1D vectors, torch.matmul supports matrix multiplication and batched operations across higher-dimensional tensors (Rank 1 and above).

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 3:**</span>

>**Create a tensor of shape `(4,4)` with values from 1 to 16 and extract:**
>    - **The second raw.**
>    - **The third column.**
>    - **The `2x2` subtensor of the bottom-right corner.**

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 4:**</span>
>**Create a tensor of shape `(2,3,4)`, reshape it to `(6,4)`, then to `(24,)` and back to `(2,3,4)`.**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 5:**</span>
>**Create a tensor from the Python list [1,2,3,4,5]. Then, create the same tensor from a NumPy array. Convert the tensor back to NumPy. Do they share memory?**

<span style='color:rgb(24, 184, 212)'>**Comment:**</span> Note that `.numpy()` and `torch.from_numpy()` create a view of the original data, meaning both objects share the same underlying memory buffer (mutating one affects the other). Conversely, the factory function `torch.tensor()` forces a strict copy of the data into a new memory location.


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 6:**</span>
>**Create the following tensors:**

>- **A 5x5 identity matrix using `torch.eye()`.**
>- **A tensor of uniformly spaced values between 0 and 1 using `torch.linspace()`.**
>- **A tensor of logarithmically spaced values using `torch.logspace()`.**


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 7:**</span>
>**Create an integer tensor (int32), a float tensor (float64), and a boolean tensor. Cast them between each other using .to() or .float(). What happens when you cast a boolean to an integer?**


<span style='color:rgb(161, 60, 219)'>**Question:**</span> *What do the numeric suffixes in data types (e.g., 8, 16, 32, 64) represent?*

They represent the number of bits allocated to store the data. For example, int8 means that each integer requires 8 bits (1 byte) of memory. Larger bit sizes provide a wider range of representable values (for integers) or higher mathematical precision (for floating-point numbers), but they also consume more memory space and can increase processing time.

<span style='color:rgb(161, 60, 219)'>**Question:**</span> *What is the significance of the 'u' prefix in primitive data types like uint8?*

The "u" stands for unsigned. It indicates that the data type cannot store negative numbers; it is exclusively restricted to zero and positive integers.

<span style='color:rgb(24, 184, 212)'>**Comment:**</span> Notice that previous outputs didn't explicitly show the tensor's dtype. PyTorch omits this information in the console for default types (`float32` for floating-point numbers and `int64` for integers). Now that a non-default type was explicitly specified, it is displayed. Note: Be cautious when mixing frameworks, as NumPy defaults to float64 instead of float32

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 8:**</span>
>**Given a random tensor of shape (6, 6), compute its mean, standard deviation, minimum, maximum, and their respective indices using argmin and argmax. Then, repeat these calculations across its rows and columns (dim=0 and dim=1).**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 9:**</span>
>**Create a tensor of shape (3, 4, 5) and practice using sum(), mean(), and max() by specifying different dimensions (dim=0, dim=1, dim=2). Observe how the shape of the result changes.**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 10:**</span>
> **Create a 2D tensor with random values and transpose it. Then, manually standarize each row so that every row has mean 0 and std 1.**

To manually standarize an element, we must compute:
$$
z = \frac{x - \mu}{\sigma}
$$

being $\mu$ the mean and $\sigma$ the standard deviation.

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 11:**</span>
>**Given two random integer tensors with values from 1 to 10, create boolean masks for: elements greater than 5, and elements that are equal in both tensors. Use these masks to filter values via boolean indexing.**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 12:**</span>
>**Create a tensor of shape (3, 1, 4, 1, 2). Remove all dimensions of size 1 using squeeze(). Then, add dimensions at different positions using unsqueeze(). What is the practical use case for this?**

The primary purpose of manipulating dimensions of size 1 via squeeze() and unsqueeze() is to guarantee strict topological alignment between your data structures and the rigid dimensional requirements of neural network layers, loss functions, and broadcasting operations. These operations add no actual data volume, but they help your data fulfill the dimensional requirements of the deep learning structures. 

For example, a Conv2d layer always expects data of shape (B, C, H, W) $\Rightarrow$ If you have only one image (C, H, W), you must use unsqueeze(0) to inject one extra dimension at the beginning, i.e., (1, C, H, W), in order to use the `Conv2D` layer.

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 13:**</span>
>**Create three tensors of shape (2, 4) and explore the differences between:**
>    - **torch.cat() along different axes.**
>    - **torch.stack() — what new dimension does it create?**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 14:**</span>
>**Create a mock image tensor of shape (3, 32, 64) (channels, height, width). Convert it to the (64, 3, 32) format using permute(). Then, practice using transpose().**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 15:**</span>
>**Without using loops, add a 1D tensor of shape (3,) to a matrix of shape (4, 3). Then, attempt to concat a (3,) tensor to a (4,) tensor — what error do you get and why? How would you fix it?(Recommendation: use the `try`flux controller)**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 16:**</span>
>**Using broadcasting and no loops, generate the multiplication table from 1 to 10 as a (10, 10) tensor. Hint: you will need a column vector and a row vector.**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 17:**</span>
>**Create a tensor of shape (4, 4) and derive a new tensor using .view(16), and another using .clone(). Modify the original tensor and observe which derived tensor changes. Next, attempt to use .view(16) on a non-contiguous tensor — what error is raised? Resolve it using .contiguous().**

A tensor is considered contiguous when its logical structure—how elements are indexed across dimensions—perfectly matches the sequential, flat layout of those elements in physical memory (RAM or VRAM). Conversely, a tensor becomes non-contiguous when metadata operations like `.T` or `.permute()` alter its logical shape by modifying the stride values without actually moving the underlying bytes. Consequently, reading a non-contiguous tensor forces the hardware pointer to jump across non-adjacent memory addresses, breaking the strict linear alignment required by high-efficiency restructuring methods like `.view()`.

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 18:**</span>
>**Create a 5x5 dense tensor initialized with zeros, then inject random values into 5 unique random positions. Convert this dense tensor to a sparse representation using .to_sparse(). What computational and memory advantages do sparse tensors provide?**

They are useful in situations where the data is sparse (for example in Conway's Game of Life States).

- **Memory Efficiency:** Bypasses allocating RAM for empty space by exclusively storing non-zero values and their spatial indices. (Note: This only yields benefits at extreme sparsity ratios, typically >90% zeros, due to the coordinate storage overhead).

- **Computational Speedup:** Specialized sparse operators (e.g., torch.smm) skip floating-point operations involving zeros, slashing execution time.



##### 🔢<span style='color:rgb(10,110,217)'>**Exercise 19:**</span>

>**Generate two random tensors of shape (3, 3) without setting a seed — do they match? Now, call torch.manual_seed(42) before each generation and check again. Write a small experiment: generate 5 pairs of random tensors (with and without a fixed seed). Why is reproducibility critical when training neural networks?**

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 20:**</span>
>**Generate a tensor of shape (100, 5) simulating a dataset of 100 samples with 5 features. For each feature, compute the mean, standard deviation, minimum, and maximum. Normalize the dataset using min-max scaling relying exclusively on tensor operations (no loops).**

### 🎌 <span style='color:rgb(10,110,217)'><u>**Summary**</u></span>

The topics covered in this notebook constitute the essential foundation for every deep learning workflow in PyTorch. A solid understanding of tensor ranks, data types, shape manipulation, memory semantics, broadcasting, and normalization is a prerequisite for correctly building, debugging, and optimizing neural network architectures.